In [ ]:
%pip install scikit-learn 
%pip install torch torchvision torchaudio
%pip install seaborn

# Introduction to Machine Learning -- MC886
# Assignment 2 - Model Selection and Regularization

**Professor:** Marcelo da Silva Reis  
**PED:** Heigon Soldera

In [ ]:
# Please, enter your name and RA number in the lines below:
print("Ana Carolina de Almeida Cardoso", "|", "246914")
print("Francisco Vinicius Guedes", "|", "260440")
print("Pedro Damasceno Vasconcellos", "|", "260640")  # You may remove this line if you are only 2 students in the group

---

## Objective

This assignment investigates the effects of overfitting in different data regimes
and test conditions.
The goal is to understand the different results of regularization when data
availability changes, and how they impact the generalization across **three**
different distributions: the one used for training, the clean one, and a
corruption the model has never seen.

### Guidelines

> Please read these guidelines carefully.

1. Questions can be answered in English or in Portuguese.

2. Use data visualization as a primary tool for understanding the problem. Inspect the training data, corrupted inputs, and model behavior. When an assumption is uncertain, verify it empirically through plots.

3. The provided code serves only as a reference. Modifications are allowed, including changes to functions and the introduction of new components. Code clarity and organization are required.

4. The use of generative AI tools (e.g., ChatGPT, GitHub Copilot, DeepSeek, LLaMA) is permitted strictly for implementation support. The use of AI for interpreting, analyzing, or discussing experimental results is discouraged. Submissions must include:
   - A declaration of which models were used.
   - The prompts that generated any incorporated code.

5. Do not hesitate to reach out to the PED/PAD via Discord.

### About the Dataset

This assignment uses the [MNIST](https://web.archive.org/web/20240213122836/http://yann.lecun.com/exdb/mnist/) dataset, a standard benchmark for image classification. It consists of 28×28 grayscale images of handwritten digits (0–9), where each pixel encodes an intensity value in [0, 255].

MNIST is useful here because it is simple enough that a linear model can do well on it, which means the learned weights are directly interpretable as images, where each weight corresponds to the influence of each pixel.

To study robustness and distribution shift, the images were corrupted with one of the corruptions of [MNIST-C](https://github.com/google-research/mnist-c) (Mu & Gilmer, 2019). Which corruption it is, is for you to find out in Part 1.

The training set consists entirely of corrupted images. The test set contains **three splits** of the same 10,000 test images:
- `test_corrupted`: corrupted test images (same corruption as training).
- `test_clean`: the original, uncorrupted MNIST test images.
- `test_impulse`: the test images with a **different** corruption, impulse (salt-and-pepper) noise, never seen during training.

Performance on `test_corrupted` measures how well the model learned the training task.
Performance on `test_clean` measures how well the model learned digit recognition.
Performance on `test_impulse` measures whether the robustness acquired during training transfers to a corruption the model was never exposed to.

Your job is to train three different models, each one with different number of
available samples:
- Full-data (60000 samples)
- Partial-data (10000 samples)
- Scarce-data (1000 samples)

Finding the best combination of hyper-parameter for each one of them.

In [ ]:
from typing import Any

from numpy.typing import NDArray
import numpy as np
import pandas as pd
import random

import torch
import torch.nn as nn
from torch import optim

import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility.
# Do not tune this number
# Re-run the notebook from the beginning before submitting.
seed = 42
torch.random.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
# The dataset is available in the classroom assignment.
# Ensure you have downloaded it and placed it alongside this notebook.

# DO NOT change this cell

def subsample_dataset(
    X: NDArray[Any], y: NDArray[Any], n: int
) -> tuple[NDArray[Any], NDArray[Any]]:
    n_samples = len(y)

    idxs = np.random.choice(
        n_samples, size=n, replace=False
    )

    return X[idxs], y[idxs]

DATA_DIR = "data_2s2026"

X_full = np.load(f"{DATA_DIR}/train/train_images.npy")
y_full = np.load(f"{DATA_DIR}/train/train_labels.npy")

X_partial, y_partial = subsample_dataset(X_full, y_full, 10_000)
X_scarce, y_scarce = subsample_dataset(X_full, y_full, 1_000)

print(f"Full-data set: {X_full.shape[0]} samples")
print(f"Partial-data set: {X_partial.shape[0]} samples")
print(f"Scarce-data set: {X_scarce.shape[0]} samples")

In [ ]:
n_cols = 5
n_rows = 2

fig, axs = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(2*n_cols, 2*n_rows))

fig.suptitle("Corrupted MNIST")

for i in range(n_rows):
    for j in range(n_cols):
        ax = axs[i, j]
        idx = random.randint(0, len(X_full)-1)

        ax.set_title(f"Sample {idx}, target {y_full[idx]}", fontsize=8)
        ax.imshow(X_full[idx], cmap='grey')
        ax.set_axis_off()

---

## Part 1. Data Analysis and Preprocessing

### 1.1 Exploring the corruption

**(1.0 point)**

Before any modeling, understand what the corruption actually does to the data.

Is the corruption random per-pixel noise, or is it a structured perturbation?
Explore the **training** dataset, understand the distribution and the effects of the introduced
noise. Useful tools are per-class mean images, per-pixel statistics (e.g., how often each
pixel is saturated at 255), and looking at many individual samples side by side.

In [ ]:
data_regimes = {
    "Scarce": (X_scarce, y_scarce),
    "Partial": (X_partial, y_partial),
    "Full": (X_full, y_full),
}

fig, axs = plt.subplots(len(data_regimes), 10, figsize=(4*len(data_regimes), 4))
for digit in range(10):
    axs[0, digit].set_title(f"digit {digit}", fontsize=8)

for i, name in enumerate(data_regimes):
    X, y = data_regimes[name]


    for digit in range(10):
        mask = (y == digit)
        mean_img = X[mask].mean(axis=0)
        axs[i, digit].imshow(mean_img, cmap="gray", vmin=0, vmax=255)
        axs[i, digit].axis("off")

    axs[i, 0].set_ylabel(f"{name}-data regime", fontsize=9)

plt.suptitle("Per-class mean images in corrupted training data", fontsize=13)
plt.tight_layout()
plt.show()

#### Corrupted samples by digit class

In [ ]:
# ============================================================
# Check random corrupted samples by digit class
# ============================================================

fig, axs = plt.subplots(10, 5, figsize=(8, 16))

for digit in range(10):

    idx = np.where(y_full == digit)[0]
    chosen = np.random.choice(idx, size=5, replace=False)

    for j, sample_idx in enumerate(chosen):

        axs[digit, j].imshow(
            X_full[sample_idx],
            cmap="gray",
            vmin=0,
            vmax=255
        )

        axs[digit, j].axis("off")

        if j == 0:
            axs[digit, j].set_ylabel(
                f"Digit {digit}",
                fontsize=10
            )

plt.suptitle("Individual corrupted samples by digit class")
plt.tight_layout()
plt.show()

#### Per-pixel mean and variance maps

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

regimes = {
    "Scarce": (X_scarce, y_scarce),
    "Partial": (X_partial, y_partial),
    "Full": (X_full, y_full)
}

# ============================================================
# Compute mean and variance maps beforehand
# ============================================================

mean_maps = {}
var_maps = {}

for regime_name, (X, y) in regimes.items():

    mean_maps[regime_name] = []
    var_maps[regime_name] = []

    for digit in range(10):

        X_digit = X[y == digit]

        mean_maps[regime_name].append(
            X_digit.mean(axis=0)
        )

        var_maps[regime_name].append(
            X_digit.var(axis=0)
        )


# ============================================================
# Common variance scale across ALL regimes and digits
# ============================================================

var_vmax = max(
    var_img.max()
    for regime_name in regimes
    for var_img in var_maps[regime_name]
)


# ============================================================
# Figure layout
#
# Rows:
# 0 -> Scarce title
# 1 -> Scarce mean
# 2 -> Scarce variance
#
# 3 -> Partial title
# 4 -> Partial mean
# 5 -> Partial variance
#
# 6 -> Full title
# 7 -> Full mean
# 8 -> Full variance
#
# 9 -> colorbar
# ============================================================

fig = plt.figure(figsize=(20, 14))

gs = fig.add_gridspec(
    nrows=10,
    ncols=10,
    height_ratios=[
        0.20, 1, 1,
        0.20, 1, 1,
        0.20, 1, 1,
        0.15
    ],
    hspace=0.18,
    wspace=0.12
)


regime_rows = {
    "Scarce": (0, 1, 2),
    "Partial": (3, 4, 5),
    "Full": (6, 7, 8)
}

last_im = None


# ============================================================
# Plot each regime
# ============================================================

for regime_name in regimes:

    title_row, mean_row, var_row = regime_rows[regime_name]

    # --------------------------------------------------------
    # Regime title
    # --------------------------------------------------------

    title_ax = fig.add_subplot(gs[title_row, :])

    title_ax.text(
        0.5,
        0.5,
        f"{regime_name}: per-class mean and variance maps",
        ha="center",
        va="center",
        fontsize=14
    )

    title_ax.axis("off")


    # --------------------------------------------------------
    # Mean and variance maps
    # --------------------------------------------------------

    for digit in range(10):

        # --------------------
        # Mean
        # --------------------

        ax_mean = fig.add_subplot(gs[mean_row, digit])

        ax_mean.imshow(
            mean_maps[regime_name][digit],
            cmap="gray",
            vmin=0,
            vmax=255
        )

        ax_mean.set_title(
            f"Digit {digit}",
            fontsize=9,
            pad=4
        )

        ax_mean.set_xticks([])
        ax_mean.set_yticks([])

        for spine in ax_mean.spines.values():
            spine.set_visible(False)


        # --------------------
        # Variance
        # --------------------

        ax_var = fig.add_subplot(gs[var_row, digit])

        last_im = ax_var.imshow(
            var_maps[regime_name][digit],
            cmap="viridis",
            vmin=0,
            vmax=var_vmax
        )

        ax_var.set_xticks([])
        ax_var.set_yticks([])

        for spine in ax_var.spines.values():
            spine.set_visible(False)


        # Row labels only on the first digit
        if digit == 0:

            ax_mean.set_ylabel(
                "Mean",
                fontsize=11,
                rotation=90,
                labelpad=8
            )

            ax_var.set_ylabel(
                "Variance",
                fontsize=11,
                rotation=90,
                labelpad=8
            )


# ============================================================
# Dedicated colorbar axis
# ============================================================

cax = fig.add_subplot(gs[9, 3:7])

cbar = fig.colorbar(
    last_im,
    cax=cax,
    orientation="horizontal"
)

cbar.set_label(
    "Pixel variance",
    fontsize=10
)

plt.show()

#### Pixel intensity distribution

In [ ]:
# ============================================================
# Non-zero pixel intensity distributions by data regime
# ============================================================

regimes = {
    "Scarce": X_scarce,
    "Partial": X_partial,
    "Full": X_full
}

fig, axs = plt.subplots(
    1, 3,
    figsize=(18, 5),
    sharex=True,
    sharey=True
)

bins = np.arange(1, 257) - 0.5

for ax, (regime_name, X) in zip(axs, regimes.items()):

    pixel_values = X.ravel()

    # Remove background pixels
    nonzero_pixels = pixel_values[pixel_values > 0]

    ax.hist(
        nonzero_pixels,
        bins=bins,
        density=True,
        histtype="stepfilled",
        alpha=0.7
    )

    ax.set_title(
        f"{regime_name}: pixel intensity distribution",
        fontsize=12
    )

    ax.set_xlabel("Pixel intensity")
    ax.set_xlim(1, 255)

    ax.grid(alpha=0.2)

# Shared y-axis label
axs[0].set_ylabel("Density")

fig.suptitle(
    "Non-zero pixel intensity distributions across data regimes",
    fontsize=14
)

plt.tight_layout()
plt.show()

#### Per-pixel saturation frequency maps across data regimes

In [ ]:
# ============================================================
# Per-pixel saturation frequency maps across data regimes
# ============================================================

regimes = {
    "Scarce": X_scarce,
    "Partial": X_partial,
    "Full": X_full
}

fig, axs = plt.subplots(
    1, 3,
    figsize=(15, 5)
)

# Compute all maps first so they share the same scale
sat_maps = {
    name: (X == 255).mean(axis=0)
    for name, X in regimes.items()
}

sat_vmax = max(
    sat_map.max()
    for sat_map in sat_maps.values()
)

for ax, (name, sat_map) in zip(axs, sat_maps.items()):

    im = ax.imshow(
        sat_map,
        cmap="hot",
        vmin=0,
        vmax=sat_vmax
    )

    ax.set_title(
        f"{name}: per-pixel saturation frequency",
        fontsize=11
    )

    ax.axis("off")

# Shared colorbar
cbar = fig.colorbar(
    im,
    ax=axs,
    orientation="horizontal",
    fraction=0.05,
    pad=0.10
)

cbar.set_label("Fraction of samples with pixel = 255")

plt.show()

#### Number of saturated pixels per image

In [ ]:
# ============================================================
# Number of saturated pixels per image across data regimes
# ============================================================

regimes = {
    "Scarce": X_scarce,
    "Partial": X_partial,
    "Full": X_full
}

fig, axs = plt.subplots(
    1, 3,
    figsize=(18, 5),
    sharex=True,
    sharey=True
)

for ax, (name, X) in zip(axs, regimes.items()):

    n_saturated = (X == 255).sum(axis=(1, 2))

    ax.hist(
        n_saturated,
        bins=30
    )

    ax.set_title(
        f"{name}: saturated pixels per image"
    )

    ax.set_xlabel(
        "Number of pixels equal to 255"
    )

axs[0].set_ylabel("Number of images")

plt.tight_layout()
plt.show()

#### Extreme-pixel fraction per image

In [ ]:
# ============================================================
# Extreme-pixel fraction per image
# ============================================================

regimes = {
    "Scarce": X_scarce,
    "Partial": X_partial,
    "Full": X_full
}

extreme_fraction_data = []

for name, X in regimes.items():

    # Convert to NumPy if X is a PyTorch tensor
    if torch.is_tensor(X):
        X_np = X.detach().cpu().numpy()
    else:
        X_np = np.asarray(X)

    # Flatten each image: (N, ...) -> (N, number_of_pixels)
    X_flat = X_np.reshape(len(X_np), -1)

    # Fraction of pixels close to either extreme, per image
    extreme_fraction = (
        (X_flat <= 10) | (X_flat >= 245)
    ).mean(axis=1)

    extreme_fraction_data.append(extreme_fraction)


plt.figure(figsize=(7, 5))

plt.boxplot(
    extreme_fraction_data,
    tick_labels=list(regimes.keys())
)

plt.title("Extreme-pixel fraction per image")
plt.xlabel("Data regime")
plt.ylabel("Fraction of extreme pixels")

plt.show()

#### Per-image mean and variance

In [ ]:
# ============================================================
# Per-image mean and variance across data regimes
# ============================================================

regimes = {
    "Scarce": X_scarce,
    "Partial": X_partial,
    "Full": X_full
}

image_mean_data = []
image_variance_data = []

for name, X in regimes.items():

    # Convert to NumPy if needed
    if torch.is_tensor(X):
        X_np = X.detach().cpu().numpy()
    else:
        X_np = np.asarray(X)

    # Flatten each image
    X_flat = X_np.reshape(len(X_np), -1)

    # One mean and one variance value per image
    image_mean = X_flat.mean(axis=1)
    image_variance = X_flat.var(axis=1)

    image_mean_data.append(image_mean)
    image_variance_data.append(image_variance)


# ============================================================
# Plot side by side
# ============================================================

fig, axs = plt.subplots(
    1, 2,
    figsize=(12, 5)
)

labels = list(regimes.keys())


# ------------------------------------------------------------
# Per-image mean
# ------------------------------------------------------------

axs[0].boxplot(
    image_mean_data,
    tick_labels=labels
)

axs[0].set_title("Per-image mean pixel intensity")
axs[0].set_xlabel("Data regime")
axs[0].set_ylabel("Mean pixel intensity")


# ------------------------------------------------------------
# Per-image variance
# ------------------------------------------------------------

axs[1].boxplot(
    image_variance_data,
    tick_labels=labels
)

axs[1].set_title("Per-image pixel variance")
axs[1].set_xlabel("Data regime")
axs[1].set_ylabel("Pixel variance")


fig.suptitle(
    "Per-image statistics across data regimes",
    fontsize=14
)

plt.tight_layout()
plt.show()

> **Question 1.1**
>
> (a) Describe the corruption. Is it random and per-sample, or is it systematic and shared across images? Does it vary across digit classes?
>
> (b) Why does the corruption turn into a faint, nearly uniform background in the per-class mean images? What does that tell you about using mean images to diagnose a corruption?
>
> (c) The corruption is made of strokes, and digits are also made of strokes. Which digits do you expect to be most easily confused with (or by) the corruption?
>
> (d) Are there data regimes where the corruption can affect the model more than in others?
> Hypothesize how these different regimes will affect a model trained on them.

**(a)** The corruption appears to be random at the sample level, but structured within each sample. It is not independent per-pixel noise: the added artifacts form coherent line-like strokes. However, their locations and orientations vary across images. Because the corruption is not spatially aligned from one sample to another, it does not produce a strong fixed pattern in the class averages. 

These is no strong evidence that the corruption mechanism itself is class-dependent. The same type of added strokes can be observed for all digit classes. The per-class mean and variance maps are also very similar across the three data regimes. Differences among classes are mainly due to the geometry of the digits themselves than an obviously different corruption process.

The distribution of pixel intensities, extreme-pixel fractions, per-image means, and per-image variance are also remarkably similar across the scarce, partial, and full regimes, which further suggests that the datasets are sampled from essentially the same corrupted data-generating process.

---

**(b)** The corruption becomes a faint background because the added strokes vary in position and orientation across samples, so they are diluted when images are averaged. In contrast, the digit structure is more consistently aligned and therefore remains visible. This shows that mean images alone can hide structured, sample-specific corruption, so individual samples and statistics such pixel-wise variance are also needed. 

--- 
**(c)** The digits that should be more vulnerable are those whose identity depends strongly on a small number of local strokes, because an additional artificial stroke can create or supress a feature that is important for distinguishing them from another class. 

Particularly, digits such as 1, 4, 7, and 9 may be more vulnerable because their identity depends strongly on a few strokes, making them easier to confuse when extra stroke-like artifacts are added. However, this is only a hypothesis from the exploratory analysis. It should later be checked against per-class classification accuracies. 

---
**(d)** The exploratory analysis suggests that the corruption distribution itself is very similar across scarce, partial, and full regimes because the smaller splits are subsamples of the full data. However, the main difference is instead the amount of available training data.

- *Scarce regime*: it should be the most affected. With few training examples, the model has fewer opportunities to observe that the corrupting strokes occur at inconsistent positions and are unrelated to the class label. This makes the model more likely to overfit due to accidental associations between particular corruption patterns and particular digits.
- *Partial regime*: the effect seen in the scarce regime will probably be reduced, since the model observes a broader range of corruption realizations. Spurious patterns are less likely to remain consistently associated with a class, although overfitting may still occur for sufficiently flexible models.
- *Full regime*: it should be the most robust. With many examples, the sample-specific corruption should average out statistically, while features that consistently characterize each digit remain predictive. Thus, the model has more evident to separate stable digit structure from unstable corruption. 

### 1.2 Preprocessing

**(1.0 point)**

Before training, the raw data must be brought into a form suitable for gradient-based optimization.

Two considerations are important here:

1. **Scale.** Raw pixel values are in [0, 255]. Gradient descent is sensitive to the scale of inputs; large input values produce large gradients, which can destabilize training.

2. **Data leakage.** Any statistics used for preprocessing must be computed from the training set only, then applied to all other splits.
Computing statistics on the full dataset, including test data, leaks information about the test distribution into preprocessing and invalidates evaluation.

In [ ]:
def preprocess_images(X: np.ndarray) -> np.ndarray:
    """
    Convert images to float32, flatten them, and rescale
    pixel values from [0, 255] to [0, 1].

    Parameters
    ----------
    X : np.ndarray
        Image array of shape (n_samples, height, width)
        or (n_samples, channels, height, width).

    Returns
    -------
    np.ndarray
        Flattened and scaled array of shape
        (n_samples, n_features).
    """
    X = np.asarray(X, dtype=np.float32)

    # Flatten each image
    X = X.reshape(X.shape[0], -1)

    # Fixed scaling using the known pixel range
    X = X / 255.0

    return X

# ============================================================
# Preprocess all training regimes
# ============================================================
X_scarce_scaled = preprocess_images(X_scarce)
X_partial_scaled = preprocess_images(X_partial)
X_full_scaled = preprocess_images(X_full)

datasets = {
    "Scarce": X_scarce_scaled,
    "Partial": X_partial_scaled,
    "Full": X_full_scaled
}

for name, X in datasets.items():
    print(
        f"{name:<8} | "
        f"min = {X.min():.4f} | "
        f"max = {X.max():.4f} | "
        f"shape = {X.shape} | "
        f"dtype = {X.dtype}"
    )

> **Question 1.2**
>
> Describe the preprocessing choices made above. What strategy have you chosen, and why?

We converted the images to `float32`, flattened each $28 \times 28 $ image into a feature vector, and rescaled pixel intensities from $[0, 255]$ to $[1, 0]$ by dividing by 255.

This strategy is well suited to multinomial logistic regression because it improves the numerical behavior of gradient-based optimization and keeps all pixel features on a common scale, which is particularly important when coefficient regularization is applied. Using a fixed division by 255 also preserves the relative structure of the corrupted images, keeps preprocessing identical across data regimes, and does not require estimating statistics from validation or test data. No denoising was applied, since the goal is to evaluate how regulariation handles the corruption rather than remove if beforehand.

---

## Part 2. Multinomial Logistic Regression

### 2.1 The model

Multinomial logistic regression (also called softmax regression) is the natural extension of binary logistic regression to $K > 2$ classes.

Given an input $\mathbf{x} \in \mathbb{R}^d$, the model computes a score (logit) for each class $k$:

$$z_k = \mathbf{w}_k^\top \mathbf{x} + b_k$$

Each logit is an unconstrained real number, in order to transform these values into
probabilities, where $p_k \geq 0$ and $\sum_k p_k = 1$, the softmax function
is used:

$$P(y = k \mid \mathbf{x}) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

Training minimizes the **cross-entropy loss**, which is the negative log-likelihood of the true class under this distribution:

$$\mathcal{L}(\mathbf{W}, \mathbf{b}) = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i \mid \mathbf{x}_i)$$

For MNIST with $d = 784$ pixels and $K = 10$ classes, the weight matrix $\mathbf{W} \in \mathbb{R}^{10 \times 784}$ has one row per class. Each row $\mathbf{w}_k$ can be reshaped into a 28×28 image and visualized directly: positive values indicate pixels whose brightness increases the log-odds of class $k$, whereas negative values suppress it.
This direct interpretability is one reason we use a linear model here.

### 2.2 Regularization

Standard cross-entropy minimization has no preference over the parameter space.
With enough capacity, or with corrupted data that contains spurious features, the model can assign large weights to uninformative or noisy pixels if doing so reduces training loss.

Regularization considers an additional assumption over the nature of the learned weights,
adding a penalty on the weights to the loss:

$$\mathcal{L}_{\text{reg}} = \mathcal{L} + \lambda \cdot \Omega(\mathbf{W})$$

Two common choices for $\Omega$:

- **L2 (Ridge):** $\Omega(\mathbf{W}) = \|\mathbf{W}\|_2^2 = \sum_{k,j} w_{kj}^2$. Penalizes large weights uniformly, where all weights shrink toward zero but none are forced exactly to zero.
In PyTorch's optimizers, L2 is implemented as `weight_decay`.

- **L1 (Lasso):** $\Omega(\mathbf{W}) = \|\mathbf{W}\|_1 = \sum_{k,j} |w_{kj}|$. Penalizes the sum of absolute values, producing sparse solutions where many weights are driven to exactly zero.

The hyperparameter $\lambda \geq 0$ controls the strength of regularization.
$\lambda = 0$ recovers the unregularized problem.

**(2.0 point)**

Implement the Multinomial Logistic Regression model using PyTorch's nn.Module
for the linear model (image as input and logits as outputs), and SGD with both
L1 and L2 regularization methods.

In [ ]:
# ============================================================
# Multinomial Logistic Regression
# ============================================================

class LinearModel(nn.Module):
    def __init__(self, n_features: int = 784, n_classes: int = 10):
        super().__init__()

        self.linear = nn.Linear(
            in_features=n_features,
            out_features=n_classes
        )

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        """
        Compute one logit for each class.
        
        Parameters
        ----------
        X : torch.Tensor
            Input tensor of shape (n_samples, n_features).

        Returns
        -------
        torch.Tensor
            Logits of shape (n_samples, n_classes).
        """
        return self.linear(X)
    

def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    """Fraction of correctly predicted samples."""
    y_pred = logits.argmax(dim=-1)
    return (y == y_pred).float().mean().item()


def train(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    n_iter: int,
    lr: float = 0.1,
    l1: float = 0.0,
    l2: float = 0.0
) -> list[float]:
    """
    Train a multinomial logistic regression model using
    full-batch SGD with L1 and/or L2 regularization.

    Parameters
    ----------
    model : nn.Module
        Multinomial logistic regression model.

    X : torch.Tensor
        Input features of shape (n_samples, n_features).

    y : torch.Tensor
        Class labels of shape (n_samples,).

    n_iter : int
        Number of optimization iterations.

    lr : float
        Learning rate.

    l1 : float
        L1 regularization strength.

    l2 : float
        L2 regularization strength.

    Returns
    -------
    list[float]
        Training loss at each iteration.
    """
    
    # CrossEntropyLoss receives raw logits directly
    criterion = nn.CrossEntropyLoss()

    # L2 regularization through weight decay
    optimizer = optim.SGD(
        model.parameters(),
        lr=lr,
        weight_decay=l2
    )

    losses = []
    accuracies = []

    for _ in range(n_iter):

        # Clear gradients from previous iteration
        optimizer.zero_grad()

        # Forward pass
        logits = model(X)

        # Standard cross-entropy loss
        loss = criterion(logits, y)

        # L1 penalty on the weight matrix only
        if l1 > 0:
            l1_penalty = model.linear.weight.abs().sum()
            loss = loss + l1 * l1_penalty

        # Backpropagation
        loss.backward()

        # SGD update
        optimizer.step()

        losses.append(loss.item())

        with torch.no_grad():
            logits_after_update = model(X)
            acc = accuracy(logits_after_update, y)
            accuracies.append(acc)

    return losses, accuracies

### 2.3 Baseline mode

**(2.0 points)**

Train a unregularized model on each training set and observe the training loss
per iteration on each model during training.

To diagnose and understand what the trained model have learned, we visualize the
learned weights as a heatmap.

In [ ]:
# Visualize learned weight images, one per digit class.
# Use a diverging colormap: red = positive weight (pixel increases log-odds),
# blue = negative weight (pixel suppresses log-odds).

def plot_weights(weights: NDArray[np.float32], title: str) -> None:
    fig, axs = plt.subplots(2, 5, figsize=(12, 5))
    fig.suptitle(title, fontsize=13)

    for digit in range(10):
        ax = axs[digit // 5, digit % 5]
        ax.imshow(weights[digit].reshape(28, 28), cmap="icefire")
        ax.set_title(f"digit {digit}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Convert preprocessed data to PyTorch tensors
# ============================================================

datasets_torch = {
    "Scarce": (
        torch.tensor(X_scarce_scaled, dtype=torch.float32),
        torch.tensor(y_scarce, dtype=torch.long)
    ),
    "Partial": (
        torch.tensor(X_partial_scaled, dtype=torch.float32),
        torch.tensor(y_partial, dtype=torch.long)
    ),
    "Full": (
        torch.tensor(X_full_scaled, dtype=torch.float32),
        torch.tensor(y_full, dtype=torch.long)
    )
}

In [ ]:
# ============================================================
# Train unregularized baseline models
# ============================================================

n_iter = 200
lr = 0.1

baseline_models = {}
baseline_losses = {}
baseline_accuracies = {}

for regime_name, (X, y) in datasets_torch.items():

    # Same initialization for fair comparison
    torch.manual_seed(seed)

    model = LinearModel()

    losses, accuracies = train(
        model=model,
        X=X,
        y=y,
        n_iter=n_iter,
        lr=lr,
        l1=0.0,
        l2=0.0
    )

    # Store model and training history
    baseline_models[regime_name] = model
    baseline_losses[regime_name] = losses
    baseline_accuracies[regime_name] = accuracies

    print(
        f"{regime_name:<8} | "
        f"Final loss = {losses[-1]:.4f} | "
        f"Final training accuracy = {accuracies[-1]:.4f}"
    )

In [ ]:
# ============================================================
# Training loss and accuracy
# ============================================================

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Loss
for regime_name, losses in baseline_losses.items():
    axs[0].plot(
        range(1, len(losses) + 1),
        losses,
        label=regime_name
    )

axs[0].set_title("Training loss")
axs[0].set_xlabel("Iteration")
axs[0].set_ylabel("Loss")
axs[0].grid(alpha=0.2)
axs[0].legend()

# Accuracy
for regime_name, accuracies in baseline_accuracies.items():
    axs[1].plot(
        range(1, len(accuracies) + 1),
        accuracies,
        label=regime_name
    )

axs[1].set_title("Training accuracy")
axs[1].set_xlabel("Iteration")
axs[1].set_ylabel("Accuracy")
axs[1].grid(alpha=0.2)
axs[1].legend()

fig.suptitle(
    "Unregularized baseline: training performance",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Extract learned weights
# ============================================================

baseline_weights = {}

for regime_name, model in baseline_models.items():

    weights = (
        model.linear.weight
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    baseline_weights[regime_name] = weights

for regime_name, weights in baseline_weights.items():

    plot_weights(
        weights,
        title=f"{regime_name}: learned weights — unregularized baseline"
    )

In [ ]:
# ============================================================
# Baseline weight distributions across data regimes
# ============================================================

# Flatten all weights from each model
weight_distributions = {
    regime: model.linear.weight.detach().cpu().numpy().ravel()
    for regime, model in baseline_models.items()
}

# Common x-axis range for fair comparison
all_weights = np.concatenate(list(weight_distributions.values()))

x_min = all_weights.min()
x_max = all_weights.max()

# Same bins for all regimes
bins = np.linspace(x_min, x_max, 40)

fig, axs = plt.subplots(
    1, 3,
    figsize=(15, 4.5),
    sharex=True,
    sharey=True
)

for ax, (regime, weights) in zip(
    axs,
    weight_distributions.items()
):
    ax.hist(
        weights,
        bins=bins,
        alpha=0.8
    )

    ax.set_title(f"{regime} baseline")
    ax.set_xlabel("Weight value")
    ax.grid(alpha=0.2)

axs[0].set_ylabel("Number of weights")

fig.suptitle(
    "Unregularized baseline weight distributions",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
for regime, weights in weight_distributions.items():
    print(
        f"{regime:<8} | "
        f"mean = {weights.mean():.4f} | "
        f"std = {weights.std():.4f} | "
        f"min = {weights.min():.4f} | "
        f"max = {weights.max():.4f} | "
        f"mean |w| = {np.abs(weights).mean():.4f}"
    )

> **Question 2.3**
>
> Inspect the weight images for each baseline model. How do you explain the
> weight values in the heatmap, and what patterns do they show?
>
> Compare between the model, how are they different?

**1. Training accuracy and loss across iterations**

- The three regimes optimize smoothly: loss decreases monotonically and training accuracy increases throughout the 200 iterations.
- Scarce repates early from the other two regimes and reaches the best training fit, with final accuracy of **0,9210** and final loss of **0,3801**. 
- Partial and Full remain very c*lose throughout training, ending at **0,8665** / **0,5210** and **0,8584** / **0,5388**, respectively.


---

## Part 3. Regularization

**(2.0 points)**

We now introduce regularization and sweep over $\lambda$ values for both L1 and L2 penalties.

Your **only decision criterion** in this part is cross-validation accuracy computed on the corrupted training data, exactly what you would have in a real scenario without access to a clean or differently corrupted test set.

For each configuration (data regime x reg type x $\lambda$), use K-fold cross-validation to estimate validation accuracy (report mean **and** standard deviation across folds).
Then, for each regularizer, **select the best $\lambda$** by CV and train a final model on the full training set of that regime.

Also train and store one model for **every** $\lambda$ of the grid on the Full-data regime: they will not be used for selection, only for the diagnostic plots of Part 4.

> **Decision task** Report, per regularizer and data regimes, which $\lambda$ you selected and why.
Inspect the weight images of your chosen models and describe what you observe.

In [ ]:
# Implement K-fold cross-validation with correct pre-processing

> **Question 3**
>
> (a) As $\lambda$ increases for L2 regularization, how does CV accuracy change on each data regime?
> Are there clear optima, or does accuracy plateau? Is the difference between the best $\lambda$ and its neighbours larger than the fold-to-fold standard deviation?
>
> (b) Compare the weight images of the CV-selected L2 model, the CV-selected L1 model, and the unregularized baseline. What differences do you observe in terms of structure and sparsity?
>
> (c) Connect the weight images to the corruption structure identified in Part 1. Does the CV-selected model appear to have suppressed the corruption pattern, amplified it, or is it ambiguous from weights alone?
>
> (d) **Predict**, based on what you see in the weights and CV curves, which of your three CV-selected models (none / L2 / L1) you expect will perform best on clean images, and which on images with impulse noise. Justify your prediction before seeing any test results.

*YOUR ANSWER HERE*

---

## Part 4. Test

**(2.0 points)**

Until now, all decisions, dataset size analysis, regularization selection, were made using only the corrupted training data and cross-validation.
**Now we use the test sets** to evaluate the models you selected.

1. Evaluate all the chosen models (baseline, CV-selected L2 and L1, for each data regime) on the three test splits.
2. Using the Full-data models stored for every $\lambda$, plot the **trade-off curves**: accuracy on `test_corrupted` versus accuracy on `test_clean`, and versus `test_impulse`, one curve per regularizer, marking the CV-selected models.
3. Plot the **sparsity** (fraction of weights with $|w| < 10^{-3}$) of the L1 and L2 models against $\lambda$.
4. Compute the **per-class accuracy** on each test split for the CV-selected models and identify which digits suffer the most from each corruption.

In [ ]:
# Load the test datasets and evaluate all the chosen models for each data regime

> **Question 4**
>
> (a) Describe the shape of the trade-off curve (corrupted-test vs. clean-test). Is there a region where increasing regularization simultaneously improves both accuracies? Is there a region of genuine trade-off?
>
> (b) The CV-selected models are marked on the plot. Where do they fall relative to the trade-off frontier? What does this tell you about using cross-validation as a model selection criterion when training and deployment distributions differ?
>
> (c) Revisit your prediction from Question 3(d). Was it correct? If not, what did the weight images fail to reveal that the test results exposed?
>
> (d) Connect the sparsity plot to the out-of-distribution results. Why might L1 regularization behave differently from L2 when evaluated on clean images and on impulse noise?
>
> (e) Compare `test_clean` and `test_impulse`. Did training on one corruption make the model robust to a different one? Which data regime and which regularizer transferred best, and why?
>
> (f) Use the per-class accuracies to check your answer to Question 1.1(c). Which digits suffer the most under each corruption?
>
> (g) In a real deployment scenario, neither the clean nor the impulse test set would be available during model selection. Given only corrupted validation data, what practical strategies could a practitioner use to make a better model selection decision than simply maximizing CV accuracy?

*YOUR ANSWER HERE*

---

## Summary

Complete the table below with results from your experiments (Full-data regime). Use the best model according to each criterion.

| Model                              | $\lambda$ | CV Accuracy (mean ± std) | Test Acc (corrupted) | Test Acc (clean) | Test Acc (impulse) | Gap clean | Gap impulse |
|------------------------------------|-----------|--------------------------|----------------------|------------------|--------------------|-----------|-------------|
| No regularization (baseline)       | 0         |                          |                      |                  |                    |           |             |
| Best CV, L2                        |           |                          |                      |                  |                    |           |             |
| Best CV, L1                        |           |                          |                      |                  |                    |           |             |
| Smallest clean gap, L2             |           |                          |                      |                  |                    |           |             |
| Smallest clean gap, L1             |           |                          |                      |                  |                    |           |             |
| Smallest impulse gap, L2           |           |                          |                      |                  |                    |           |             |
| Smallest impulse gap, L1           |           |                          |                      |                  |                    |           |             |

*Gap clean = Test Acc (corrupted) - Test Acc (clean); Gap impulse = Test Acc (corrupted) - Test Acc (impulse).*

---

## Assignment Submission

This notebook must be filled with your solution and submitted on the Assignment's entry at our Google Classroom page.

Before submitting, verify that:
- The notebook runs end-to-end without errors (use **Run All** after restarting the kernel).
- All plots are rendered in the output.
- All written answers are filled in.
- The summary table is completed.
- AI usage declaration and prompts are included if applicable.

**Important: Only one member of the group should submit.**

### Policy for late submissions

You are NOT encouraged to submit after the deadline; however, in this case the following penalties apply:

- 25% deduction for 1-day late submission.
- 50% deduction for 2-day late submission.
- 75% deduction for 3-day late submission.